# Segmentation client par K-means

Ce notebook compare deux segmentations construites à partir du fichier complet produit par le notebook précédent :

- **Approche valide** : `recence_jours_scaled`, `frequence_scaled`, `montant_total_scaled` ;
- **Approche globale** : `recence_globale_jours_scaled`, `frequence_globale_scaled`, `montant_global_scaled`.

Pour chaque approche, plusieurs valeurs de `k` sont testées avec la méthode du coude et le score de silhouette. Le choix final de `k` est laissé à l'interprétation des graphiques.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
import src.clustering as clustering

importlib.reload(clustering)
from src.clustering import (
    GLOBAL_SCALED_COLUMNS,
    VALID_SCALED_COLUMNS,
    evaluate_k_values,
    run_segmentation,
)

FIGURES_DIR = Path("../figures")
OUTPUT_DIR = Path("../outputs")
FIGURES_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## 1. Chargement des caractéristiques

Le fichier contient une ligne par client, les RFM bruts, leurs transformations et les variables descriptives. K-means est entraîné uniquement sur les colonnes standardisées.

In [ ]:
input_path = Path("../data/processed/features_rfm_data.csv")
data = pd.read_csv(input_path)

required_columns = {"CustomerID", "pays_residence", "produit_top_code", "produit_top_description"}
required_columns.update(VALID_SCALED_COLUMNS + GLOBAL_SCALED_COLUMNS)
missing_columns = required_columns.difference(data.columns)
if missing_columns:
    raise ValueError(f"Colonnes manquantes : {sorted(missing_columns)}")

print(f"Clients chargés : {len(data):,}")
data.head()

## 2. Évaluation des valeurs de k

Le coude mesure la baisse de l'inertie quand `k` augmente. Le score de silhouette mesure la cohésion des segments et leur séparation ; plus il est élevé, meilleure est la séparation. Le score est calculé pour `k >= 2`.

In [ ]:
k_values = range(2, 11)

results_valid = evaluate_k_values(data, VALID_SCALED_COLUMNS, k_values)
results_global = evaluate_k_values(data, GLOBAL_SCALED_COLUMNS, k_values)

results_valid["approche"] = "Achats valides"
results_global["approche"] = "RFM global"
results = pd.concat([results_valid, results_global], ignore_index=True)

results

## 3. Graphiques de comparaison

Les deux approches sont affichées dans une même figure pour faciliter la comparaison. Le coude ne suffit pas à lui seul : le choix de `k` doit aussi tenir compte du score de silhouette et de l'interprétabilité métier.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for row, (label, approach_results) in enumerate(
    [("Achats valides", results_valid), ("RFM global", results_global)]
):
    axes[row, 0].plot(approach_results["k"], approach_results["inertie"], marker="o")
    axes[row, 0].set_title(f"Coude - {label}")
    axes[row, 0].set_xlabel("Nombre de segments (k)")
    axes[row, 0].set_ylabel("Inertie")

    axes[row, 1].plot(approach_results["k"], approach_results["silhouette"], marker="o")
    axes[row, 1].set_title(f"Silhouette - {label}")
    axes[row, 1].set_xlabel("Nombre de segments (k)")
    axes[row, 1].set_ylabel("Score de silhouette")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "evaluation_k_deux_approches.png", dpi=150)
plt.show()

## 4. Aide à la décision

Le meilleur score de silhouette est un repère statistique, pas une décision automatique. On vérifie aussi le coude et la lisibilité des profils avant de fixer les valeurs finales de `k`.

In [ ]:
best_k = results.loc[results.groupby("approche")["silhouette"].idxmax()]
print("Meilleur score de silhouette observé pour chaque approche :")
best_k[["approche", "k", "silhouette", "inertie"]]

In [ ]:
k_valid_final = 4
k_global_final = 4

summaries = run_segmentation(
    input_path=input_path,
    output_dir=OUTPUT_DIR,
    k_valid=k_valid_final,
    k_global=k_global_final,
)

for approach, summary in summaries.items():
    print(f"\nRésumé des segments - {approach}")
    display(summary)

## Fichiers produits

Le script réutilisable se trouve dans `src/clustering.py`. Il peut être relancé après modification des valeurs de `k`.

Pour chaque approche, il produit :

- `clients_segmentes_valid.csv` ou `clients_segmentes_global.csv` ;
- `resume_segments_valid.csv` ou `resume_segments_global.csv`.

Le résumé contient l'effectif, la part du chiffre d'affaires total, les moyennes RFM, le pays dominant et le produit dominant de chaque segment.